In [ ]:
from sampo.scheduler.genetic import GeneticScheduler
from LLMHeuristicScheduler.base import LLMHeuristicScheduler
from sampo_api import contractor

import logging
import re
from sampo.base import SAMPO

# Genetic Algrotihm basic SAMPO  vs Genetic Algorithm with LLMs generated init population

Цель исследования:    
    - Проверить сходимость за сколько был достигнут GAP = 0,   
    Гипотеза: Iter_LLM <  Iter_GA    
    - Оптимальность  
    Гипотеза: makespan_LLM < makespan_GA


In [ ]:
import logging
import re

class StatsCollector:
    def __init__(self,):
        self.items = [{'best_fitness': float('inf')}]

    def add(self, generation, population, best_fitness):
        self.items.append({
            "generation": generation,
            "population": population,
            "best_fitness": best_fitness,
        })
    def clear(self):
        self.items = [{'best_fitness': float('inf')}]


class StatsHandler(logging.Handler):
    pattern = re.compile(
        r"-- Generation (?P<generation>\d+), population=(?P<population>\d+), best fitness=\((?P<fitness>\d+)\.0,\) --"
    )
    def __init__(self, collector):
        super().__init__()
        self.collector = collector  # сюда кладём внешний объект
    def emit(self, record):
        msg = self.format(record)
        m = self.pattern.search(msg)
        if not m:
            return
        generation = int(m.group("generation"))
        population = int(m.group("population"))
        best_fitness = float(m.group("fitness"))
        if self.collector.items[-1]['best_fitness'] > best_fitness:
            self.collector.add(generation, population, best_fitness)

In [ ]:
from sampo.schemas.graph import WorkGraph
from collections import defaultdict
import os

def init_experiment(GA_params, model_names):
    solvers_dict = {}
    solvers_dict['genetic'] = GeneticScheduler(**GA_params)
    for model_name in model_names:
        model = LLMHeuristicScheduler(model_name, **GA_params)
        solvers_dict[model_name] = model
    return solvers_dict

def run_experiment(solvers_dict, path_dataset = 'wgs/small_synth'):
    sc = StatsCollector()
    stats_handler = StatsHandler(sc)
    SAMPO.logger.addHandler(stats_handler)
    experiment_logs = defaultdict(list)
    for f in os.listdir(path_dataset):
        wg , contractors = WorkGraph.loadf(path_dataset, f[:-5]), contractor(N=5)
        for solver, model in solvers_dict.items():
            model.schedule(wg, contractors)
            experiment_logs[solver].append(sc.items)
            sc.clear()
    return experiment_logs
    

In [4]:
from sampo.schemas.graph import WorkGraph
import os
from sampo_api import contractor

maxx, minn = -float('inf'), float('inf')
for f in os.listdir('wgs/small_synth'):
        wg , contractors = WorkGraph.loadf('wgs/small_synth', f[:-5]), contractor(N=5)
        N = len(wg.nodes) 
        maxx = max(maxx, N)
        minn = min(minn, N)

print(maxx, minn)

49 31


In [ ]:
GA_params  = {
    'number_of_generation' : 50,
    'size_of_population' : 50,
    'mutate_order' : 0.15,
    'mutate_resources': 0.1,
    'mutate_zones': 0.1}

solvers_dict = init_experiment(GA_params, ['deepseek_chat', 'deepseek_reasoner'])

history = run_experiment(solvers_dict)

# 108 m


[SAMPO] [INFO] Toolbox initialization & first population took 27.563095092773438 ms


Genetic optimizing took 27.395009994506836 ms


[SAMPO] [INFO] First population evaluation took 311.5262985229492 ms
[SAMPO] [INFO] -- Generation 1, population=50, best fitness=(68.0,) --
[SAMPO] [INFO] -- Generation 2, population=50, best fitness=(66.0,) --
[SAMPO] [INFO] -- Generation 3, population=50, best fitness=(66.0,) --
[SAMPO] [INFO] -- Generation 4, population=50, best fitness=(66.0,) --
[SAMPO] [INFO] -- Generation 5, population=50, best fitness=(66.0,) --
[SAMPO] [INFO] -- Generation 6, population=50, best fitness=(66.0,) --
[SAMPO] [INFO] -- Generation 7, population=50, best fitness=(66.0,) --
[SAMPO] [INFO] -- Generation 8, population=50, best fitness=(66.0,) --
[SAMPO] [INFO] -- Generation 9, population=50, best fitness=(66.0,) --
[SAMPO] [INFO] -- Generation 10, population=50, best fitness=(66.0,) --
[SAMPO] [INFO] -- Generation 11, population=50, best fitness=(66.0,) --
[SAMPO] [INFO] -- Generation 12, population=50, best fitness=(66.0,) --
[SAMPO] [INFO] -- Generation 13, population=50, best fitness=(66.0,) --
[SAM

Genetic optimizing took 33.14685821533203 ms


[SAMPO] [INFO] First population evaluation took 377.75182723999023 ms
[SAMPO] [INFO] -- Generation 1, population=50, best fitness=(68.0,) --
[SAMPO] [INFO] -- Generation 2, population=50, best fitness=(68.0,) --
[SAMPO] [INFO] -- Generation 3, population=50, best fitness=(65.0,) --
[SAMPO] [INFO] -- Generation 4, population=50, best fitness=(59.0,) --
[SAMPO] [INFO] -- Generation 5, population=50, best fitness=(59.0,) --
[SAMPO] [INFO] -- Generation 6, population=50, best fitness=(59.0,) --
[SAMPO] [INFO] -- Generation 7, population=50, best fitness=(59.0,) --
[SAMPO] [INFO] -- Generation 8, population=50, best fitness=(59.0,) --
[SAMPO] [INFO] -- Generation 9, population=50, best fitness=(59.0,) --
[SAMPO] [INFO] -- Generation 10, population=50, best fitness=(59.0,) --
[SAMPO] [INFO] -- Generation 11, population=50, best fitness=(59.0,) --
[SAMPO] [INFO] -- Generation 12, population=50, best fitness=(59.0,) --
[SAMPO] [INFO] -- Generation 13, population=50, best fitness=(58.0,) --
[SA

...

[SAMPO] [INFO] First population evaluation took 274.2328643798828 ms
[SAMPO] [INFO] -- Generation 1, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 2, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 3, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 4, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 5, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 6, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 7, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 8, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 9, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 10, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 11, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 12, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 13, population=50, best fitness=(57.0,) --
[SAM

Genetic optimizing took 94.85101699829102 ms


[SAMPO] [INFO] First population evaluation took 297.0149517059326 ms
[SAMPO] [INFO] -- Generation 1, population=50, best fitness=(69.0,) --
[SAMPO] [INFO] -- Generation 2, population=50, best fitness=(69.0,) --
[SAMPO] [INFO] -- Generation 3, population=50, best fitness=(69.0,) --
[SAMPO] [INFO] -- Generation 4, population=50, best fitness=(69.0,) --
[SAMPO] [INFO] -- Generation 5, population=50, best fitness=(69.0,) --
[SAMPO] [INFO] -- Generation 6, population=50, best fitness=(67.0,) --
[SAMPO] [INFO] -- Generation 7, population=50, best fitness=(67.0,) --
[SAMPO] [INFO] -- Generation 8, population=50, best fitness=(67.0,) --
[SAMPO] [INFO] -- Generation 9, population=50, best fitness=(67.0,) --
[SAMPO] [INFO] -- Generation 10, population=50, best fitness=(67.0,) --
[SAMPO] [INFO] -- Generation 11, population=50, best fitness=(67.0,) --
[SAMPO] [INFO] -- Generation 12, population=50, best fitness=(62.0,) --
[SAMPO] [INFO] -- Generation 13, population=50, best fitness=(62.0,) --
[SAM

Genetic optimizing took 103.35588455200195 ms


[SAMPO] [INFO] First population evaluation took 271.52299880981445 ms
[SAMPO] [INFO] -- Generation 1, population=50, best fitness=(64.0,) --
[SAMPO] [INFO] -- Generation 2, population=50, best fitness=(64.0,) --
[SAMPO] [INFO] -- Generation 3, population=50, best fitness=(64.0,) --
[SAMPO] [INFO] -- Generation 4, population=50, best fitness=(64.0,) --
[SAMPO] [INFO] -- Generation 5, population=50, best fitness=(64.0,) --
[SAMPO] [INFO] -- Generation 6, population=50, best fitness=(64.0,) --
[SAMPO] [INFO] -- Generation 7, population=50, best fitness=(64.0,) --
[SAMPO] [INFO] -- Generation 8, population=50, best fitness=(62.0,) --
[SAMPO] [INFO] -- Generation 9, population=50, best fitness=(62.0,) --
[SAMPO] [INFO] -- Generation 10, population=50, best fitness=(62.0,) --
[SAMPO] [INFO] -- Generation 11, population=50, best fitness=(62.0,) --
[SAMPO] [INFO] -- Generation 12, population=50, best fitness=(62.0,) --
[SAMPO] [INFO] -- Generation 13, population=50, best fitness=(62.0,) --
[SA

In [7]:
from scripts.metrics import summarize_runs_main, metrics_by_init_population, aggregate_metrics
import pandas as pd


genetic_summary = summarize_runs_main(history['genetic'])

res = {}
for solver, h in history.items():
    if solver == 'genetic':
        continue
    runs = metrics_by_init_population(genetic_summary, h)
    res[solver] = aggregate_metrics(runs)

res_df = pd.DataFrame.from_dict(res)

res_df

,deepseek_chat,deepseek_reasoner
success_rate,65.306122,75.510204
found_better_rate,48.979592,67.346939
mean_delta_gen_to_main,-3.937500,-10.135135
mean_delta_gen_to_better,-3.000000,-6.878788
mean_gain_improve,3.200000,4.200000
count_gain_improve,24.000000,33.000000


In [1]:
res_df.to_clipboard()

NameError: name 'res_df' is not defined

In [ ]:
# runs = metrics_by_init_population(
#         genetic_summary,
#         history['deepseek_reasoner'],
#     )
# runs

In [ ]:
res_df

In [ ]:
# from sampo.schemas.graph import WorkGraph
# from scripts.wg_converter import WorkGraphConverter, ProjectConverter



# wg = WorkGraph.loadf('wgs/small_synth', 'wg_11')
# work_units = {node.id: node.work_unit for node in wg.nodes}


# def read_file(path='wgs/small_synth', file='wg_11', contractors_N=5):  
#     wg = WorkGraph.loadf(path, file)
#     contractors = contractor(N = contractors_N)
#     conv = WorkGraphConverter()
#     data = conv.convert(wg, contractors)['rcpsp_data']
#     return data

# data = read_file()



# import os
# from scripts.valid import run_heuristic, validate_schedule_bool
# from sampo.schemas.schedule_spec import ScheduleSpec


# path = 'Heuristics/deepseek_chat'
# files = [f for f in os.listdir(path) if f not in ('Steps', 'original_heuristics.json')]

# pc = ProjectConverter(wg, contractor(N = 5))
# for f in files:
#     schedule, order, resource_usage, job_usage,  makespan = run_heuristic(path, f, data)
#     schedule_obj =  pc.to_schedule(schedule, order, job_usage,  makespan)

#     if validate_schedule_bool(schedule_obj, wg, contractor(N=5), ScheduleSpec()):
#         print(f, makespan)
      
    
